# Tier 2 Diagnostics — Brain Training Failure + Bone Marrow Coverage

This notebook investigates two questions before re-training:

### Part A: Is `bone_marrow.h5ad` adequate to cover lymphoid + myeloid cell types after we move `lymphoid.h5ad` and `myeloid.h5ad` to `LAB_CONFIGS`?
- Inventory leaf classes in each of the three datasets
- Cross-reference: which lymphoid/myeloid leaves are present in bone_marrow vs lost entirely?
- Cell-count adequacy per leaf (need ≥ MIN_CELLS_PER_TYPE to survive the filter)

### Part B: Why is Brain_normal zero-shot evaluation failing for both CE and HCE?
- **B1**: Gene panel format check — HGNC symbols vs Ensembl IDs vs other across brain datasets
- **B2**: Tokenizer coverage — what fraction of the top-200 expressed genes per cell tokenize cleanly in the C2S vocab?
- **B3**: Cell sentence inspection — visually compare brain_normal vs training-brain cell sentences
- **B4**: Training class counts — does the model see enough examples of each brain cell type?

This notebook is read-only against the existing data. It does not retrain anything.


## 1. Imports + config

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from collections import Counter
from transformers import AutoTokenizer

warnings.filterwarnings('ignore')

# ── Paths (match main notebook) ───────────────────────────────────────────────
PATHS = {
    'lymphoid':       'lab-data/lymphoid.h5ad',
    'myeloid':        'lab-data/myeloid.h5ad',
    'bone_marrow':    'census_data/bone_marrow.h5ad',
    'brain_glia':     'brain_new.h5ad',                  # training brain (glia)
    'brain_neurons':  'brain_neurons_processed.h5ad',    # training brain (neurons)
    'brain_normal':   'lab-data/Brain_normal.h5ad',      # lab brain (zero-shot)
}

LABEL_COLS = {
    'lymphoid':      'cell_type',
    'myeloid':       'cell_type',
    'bone_marrow':   'cell_type',
    'brain_glia':    'cell_type',
    'brain_neurons': 'label',
    'brain_normal':  'cell_type',
}

C2S_MODEL_NAME      = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'
MIN_CELLS_PER_TYPE  = 100   # same as main notebook
MAX_CELLS_PER_TYPE  = 300
TOP_K_GENES         = 200

print('[OK] Config loaded')

/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/c2s-justin/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] Config loaded


In [2]:
def is_valid(value):
    if pd.isna(value): return False
    return str(value).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

def get_gene_symbols(adata):
    if 'feature_name' in adata.var.columns:
        return np.array(adata.var['feature_name'].astype(str))
    return np.array(adata.var_names.astype(str))

def cell_to_text_dense(X_row, gene_symbols, top_k=200):
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz  = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    vals = row[nz]
    if len(nz) > top_k:
        idx = np.argpartition(vals, -top_k)[-top_k:]
        nz  = nz[idx[np.argsort(vals[idx])[::-1]]]
    else:
        nz = nz[np.argsort(vals)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)

def label_counts(adata, label_col, min_cells=MIN_CELLS_PER_TYPE):
    """Return cell-type counts filtered to valid + above-threshold types."""
    mask    = adata.obs[label_col].apply(is_valid)
    labels  = adata.obs.loc[mask, label_col].astype(str)
    counts  = labels.value_counts()
    return counts, counts[counts >= min_cells]

print('[OK] Helpers defined')

[OK] Helpers defined


## Part A — Bone marrow coverage audit

We want to know: if we remove `lymphoid.h5ad` and `myeloid.h5ad` from training, does `bone_marrow.h5ad` still expose the model to lymphoid/myeloid cell types? If not, the model's leaf vocabulary will shrink and zero-shot lymphoid/myeloid predictions will fall back to coarser labels.

In [3]:
print('Loading lymphoid / myeloid / bone_marrow ...')
adatas = {}
for name in ['lymphoid', 'myeloid', 'bone_marrow']:
    print(f'  loading {name} ({PATHS[name]}) ...')
    adatas[name] = sc.read_h5ad(PATHS[name], backed='r')
    print(f'    -> {adatas[name].n_obs:,} cells, {adatas[name].n_vars:,} genes')

print('\n--- Cell type counts (>= MIN_CELLS_PER_TYPE) ---')
type_sets = {}
for name in ['lymphoid', 'myeloid', 'bone_marrow']:
    label_col = LABEL_COLS[name]
    all_counts, kept_counts = label_counts(adatas[name], label_col)
    type_sets[name] = set(kept_counts.index)
    print(f'\n[{name}]  total types = {len(all_counts)},  after filter (>={MIN_CELLS_PER_TYPE}) = {len(kept_counts)}')
    for ct, n in kept_counts.items():
        print(f'    {ct:<60} n={n:>6}')

Loading lymphoid / myeloid / bone_marrow ...
  loading lymphoid (lab-data/lymphoid.h5ad) ...
    -> 7,173 cells, 13,993 genes
  loading myeloid (lab-data/myeloid.h5ad) ...
    -> 6,218 cells, 14,395 genes
  loading bone_marrow (census_data/bone_marrow.h5ad) ...
    -> 759,562 cells, 61,497 genes

--- Cell type counts (>= MIN_CELLS_PER_TYPE) ---

[lymphoid]  total types = 11,  after filter (>=100) = 10
    plasma_cell                                                  n=  1476
    CD4_T_cell_naive_or_memory                                   n=  1287
    CD8_T_cell_early_activated                                   n=  1040
    B_cell                                                       n=   990
    CD8_T_cell_late_exhausted                                    n=   584
    CD4_T_cell_activated                                         n=   532
    Treg_cell                                                    n=   391
    NK_cell                                                      n=   369
   

In [4]:
# ── Cross-reference: which lymphoid/myeloid leaves are also in bone_marrow? ──
print('=' * 100)
print('  LEAF COVERAGE AUDIT')
print('=' * 100)

bm = type_sets['bone_marrow']

for src in ['lymphoid', 'myeloid']:
    src_types = type_sets[src]
    overlap   = src_types & bm
    lost      = src_types - bm
    print(f'\n[{src}]  {len(src_types)} leaves total')
    print(f'  also in bone_marrow : {len(overlap):>3}  ({len(overlap)/max(1,len(src_types))*100:.0f}%)')
    print(f'  LOST if we move out : {len(lost):>3}  ({len(lost)/max(1,len(src_types))*100:.0f}%)')
    if overlap:
        print(f'  -- Overlap --')
        for ct in sorted(overlap):
            print(f'     {ct}')
    if lost:
        print(f'  -- LOST (no bone_marrow coverage) --')
        for ct in sorted(lost):
            print(f'     {ct}')

# Combined leaf set
combined_lymph_mye = type_sets['lymphoid'] | type_sets['myeloid']
print(f'\n--- Net change in training vocabulary ---')
print(f'  Leaves currently provided by lymphoid+myeloid     : {len(combined_lymph_mye)}')
print(f'  Leaves provided by bone_marrow alone              : {len(bm)}')
print(f'  Leaves in (lymphoid ∪ myeloid) but NOT bone_marrow: {len(combined_lymph_mye - bm)}')
print(f'  Leaves in bone_marrow but NOT (lymphoid ∪ myeloid): {len(bm - combined_lymph_mye)}')

  LEAF COVERAGE AUDIT

[lymphoid]  10 leaves total
  also in bone_marrow :   0  (0%)
  LOST if we move out :  10  (100%)
  -- LOST (no bone_marrow coverage) --
     B_cell
     B_cell_naive
     CD4_T_cell_activated
     CD4_T_cell_naive_or_memory
     CD8_T_cell_early_activated
     CD8_T_cell_late_exhausted
     NK_cell
     Treg_cell
     plasma_cell
     plasma_cell_proliferating

[myeloid]  13 leaves total
  also in bone_marrow :   1  (8%)
  LOST if we move out :  12  (92%)
  -- Overlap --
     neutrophil
  -- LOST (no bone_marrow coverage) --
     alveolar_macrophage
     macrophage_APOE_CHIT
     macrophage_C3
     macrophage_F13A1
     macrophage_FOLR2
     macrophage_ISG_expressing
     macrophage_VEGFA
     macrophage_glycolytic
     monocyte_AREG_EREG
     monocyte_CSF3R
     monocyte_ITGAL
     monocyte_SOCS3

--- Net change in training vocabulary ---
  Leaves currently provided by lymphoid+myeloid     : 23
  Leaves provided by bone_marrow alone              : 109
  Leaves 

In [5]:
# ── Visualize via a coverage table ────────────────────────────────────────────
all_immune_leaves = sorted(type_sets['lymphoid'] | type_sets['myeloid'] | type_sets['bone_marrow'])
def count_in(name, ct):
    label_col = LABEL_COLS[name]
    return int((adatas[name].obs[label_col].astype(str) == ct).sum())

rows = []
for ct in all_immune_leaves:
    rows.append({
        'cell_type'  : ct,
        'lymphoid'   : count_in('lymphoid', ct),
        'myeloid'    : count_in('myeloid', ct),
        'bone_marrow': count_in('bone_marrow', ct),
    })
df_cov = pd.DataFrame(rows).sort_values('cell_type').reset_index(drop=True)
df_cov['in_lymph_mye'] = (df_cov['lymphoid'] + df_cov['myeloid'] >= MIN_CELLS_PER_TYPE)
df_cov['in_bm']        = (df_cov['bone_marrow'] >= MIN_CELLS_PER_TYPE)
df_cov['status'] = np.select(
    [df_cov['in_lymph_mye'] & df_cov['in_bm'],
     df_cov['in_lymph_mye'] & ~df_cov['in_bm'],
     ~df_cov['in_lymph_mye'] & df_cov['in_bm']],
    ['covered_by_both', 'lost_if_moved', 'unique_to_bm'],
    default='neither')

print('\nLeaf-by-leaf coverage (sorted by status, then count):')
print(df_cov[['cell_type','lymphoid','myeloid','bone_marrow','status']].to_string(index=False))

print('\n--- Summary ---')
print(df_cov['status'].value_counts().to_string())


Leaf-by-leaf coverage (sorted by status, then count):
                                                                 cell_type  lymphoid  myeloid  bone_marrow          status
                                                                    B cell         0        0         2020    unique_to_bm
                                                                    B_cell       990        0            0   lost_if_moved
                                                              B_cell_naive       181        0            0   lost_if_moved
                                                    CD14-positive monocyte         0        0        34104    unique_to_bm
                                     CD14-positive, CD16-positive monocyte         0        0         4996    unique_to_bm
                     CD16-negative, CD56-bright natural killer cell, human         0        0         6524    unique_to_bm
                        CD16-positive, CD56-dim natural killer cell, human         0

### Part A interpretation

Read the coverage table above:
- **`covered_by_both`** — leaf appears in both bone_marrow and lymphoid/myeloid. Safe to move; no loss.
- **`lost_if_moved`** — leaf only appears in lymphoid/myeloid. **This is the cost of moving them to LAB_CONFIGS.** The model will no longer see these labels during training and won't be able to predict them on zero-shot data.
- **`unique_to_bm`** — leaf only in bone_marrow. Already in training; not affected.

If `lost_if_moved` is small (e.g., 0–5 leaves, mostly very specific activation/exhaustion states like `CD8_T_cell_late_exhausted`), then moving is safe. If it's large (10+ leaves of distinct cell types), you'd want to mitigate by either (a) keeping a stratified subsample of lymphoid+myeloid in training and putting the held-out portion in LAB_CONFIGS, or (b) adding more general immune leaves from CellxGene to fill the gap.

## Part B — Brain training diagnosis

The brain_normal zero-shot evaluation is fundamentally broken (~15% lineage match for both models). We need to figure out whether the cause is:

1. **Gene panel mismatch** — Brain_normal uses gene IDs/symbols the tokenizer can't read (B1, B2)
2. **Cell sentence content failure** — markers exist but get drowned in housekeeping genes (B3)
3. **Training imbalance** — model wasn't shown enough brain neurons/glia (B4)

In [6]:
# ── B1: Gene panel format inspection ──────────────────────────────────────────
print('=' * 100)
print('  B1: GENE PANEL FORMAT INSPECTION')
print('=' * 100)

for name in ['brain_glia', 'brain_neurons', 'brain_normal']:
    print(f'\n[{name}]')
    print(f'  path: {PATHS[name]}')
    if name not in adatas:
        adatas[name] = sc.read_h5ad(PATHS[name], backed='r')
    a = adatas[name]
    print(f'  n_cells  : {a.n_obs:,}')
    print(f'  n_genes  : {a.n_vars:,}')
    print(f'  var.columns: {list(a.var.columns)}')
    print(f'  var.index sample  (10): {list(a.var_names[:10])}')
    if 'feature_name' in a.var.columns:
        print(f'  feature_name sample: {list(a.var["feature_name"].astype(str)[:10])}')
    if 'feature_id' in a.var.columns:
        print(f'  feature_id sample  : {list(a.var["feature_id"].astype(str)[:10])}')
    # Heuristic: are these HGNC symbols or Ensembl IDs?
    syms = get_gene_symbols(a)
    n_ensembl = sum(1 for s in syms[:1000] if str(s).startswith('ENSG'))
    n_mt      = sum(1 for s in syms[:1000] if str(s).startswith('MT-'))
    n_rps     = sum(1 for s in syms[:1000] if str(s).startswith(('RPS','RPL')))
    print(f'  format guess (first 1000 genes): Ensembl={n_ensembl}  MT-*={n_mt}  RPS/RPL={n_rps}')

  B1: GENE PANEL FORMAT INSPECTION

[brain_glia]
  path: brain_new.h5ad
  n_cells  : 888,263
  n_genes  : 58,232
  var.columns: ['Biotype', 'Chromosome', 'End', 'Gene', 'Start', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
  var.index sample  (10): ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938', 'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084', 'ENSG00000001167']
  feature_name sample: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM', 'FGR', 'CFH', 'FUCA2', 'GCLC', 'NFYA']
  format guess (first 1000 genes): Ensembl=0  MT-*=0  RPS/RPL=4

[brain_neurons]
  path: brain_neurons_processed.h5ad
  n_cells  : 100,000
  n_genes  : 58,232
  var.columns: ['Biotype', 'Chromosome', 'End', 'Gene', 'Start', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
  var.index sample  (10): ['ENSG00000000003', 'E

In [7]:
# ── B2: Tokenizer coverage on top-K expressed genes per cell ─────────────────
print('=' * 100)
print('  B2: TOKENIZER COVERAGE ON TOP-200 EXPRESSED GENES')
print('=' * 100)
print('  Loading C2S tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
vocab = set(tokenizer.get_vocab().keys())
print(f'  vocab size = {len(vocab):,}')

def coverage_stats(name, n_sample=30):
    """For n_sample random cells, compute % of top-K expressed genes that tokenize as a single token."""
    a = adatas[name]
    gene_syms = get_gene_symbols(a)
    rng = np.random.default_rng(42)
    n_obs = a.n_obs
    sample_idx = rng.choice(n_obs, size=min(n_sample, n_obs), replace=False)

    pct_in_vocab, pct_single_tok, n_tokens_avg = [], [], []
    for i in sample_idx:
        # load row (handle backed mode by indexing X)
        row = a.X[int(i)]
        if hasattr(row, 'toarray'): row = row.toarray().flatten()
        else: row = np.array(row).flatten()
        nz = np.where(row > 0)[0]
        if len(nz) == 0:
            continue
        vals = row[nz]
        if len(nz) > TOP_K_GENES:
            sel = np.argpartition(vals, -TOP_K_GENES)[-TOP_K_GENES:]
            nz  = nz[sel[np.argsort(vals[sel])[::-1]]]
        else:
            nz = nz[np.argsort(vals)[::-1]]
        top_genes = [str(gene_syms[j]) for j in nz]

        in_vocab = sum(1 for g in top_genes if g in vocab)
        single_tok = sum(1 for g in top_genes if len(tokenizer.encode(g, add_special_tokens=False)) == 1)
        avg_tok = np.mean([len(tokenizer.encode(g, add_special_tokens=False)) for g in top_genes])

        pct_in_vocab.append(in_vocab / len(top_genes) * 100)
        pct_single_tok.append(single_tok / len(top_genes) * 100)
        n_tokens_avg.append(avg_tok)

    return {
        'n_cells_sampled': len(pct_in_vocab),
        'pct_in_vocab_exact':    float(np.mean(pct_in_vocab))   if pct_in_vocab else 0.0,
        'pct_single_token':      float(np.mean(pct_single_tok)) if pct_single_tok else 0.0,
        'avg_tokens_per_gene':   float(np.mean(n_tokens_avg))   if n_tokens_avg else 0.0,
    }

print('\nSampling 30 cells per dataset and computing tokenizer coverage on top-200 genes ...\n')
for name in ['brain_glia', 'brain_neurons', 'brain_normal']:
    stats = coverage_stats(name, n_sample=30)
    print(f'[{name}]')
    print(f'  cells sampled                       : {stats["n_cells_sampled"]}')
    print(f'  %% genes that are exact vocab tokens : {stats["pct_in_vocab_exact"]:>6.1f}')
    print(f'  %% genes encoded as single token     : {stats["pct_single_token"]:>6.1f}')
    print(f'  avg tokens per gene name             : {stats["avg_tokens_per_gene"]:>6.2f}')
    print()

  B2: TOKENIZER COVERAGE ON TOP-200 EXPRESSED GENES
  Loading C2S tokenizer ...
  vocab size = 50,277

Sampling 30 cells per dataset and computing tokenizer coverage on top-200 genes ...

[brain_glia]
  cells sampled                       : 30
  %% genes that are exact vocab tokens :    0.6
  %% genes encoded as single token     :    0.6
  avg tokens per gene name             :   3.48

[brain_neurons]
  cells sampled                       : 30
  %% genes that are exact vocab tokens :    0.2
  %% genes encoded as single token     :    0.2
  avg tokens per gene name             :   3.49

[brain_normal]
  cells sampled                       : 30
  %% genes that are exact vocab tokens :    0.9
  %% genes encoded as single token     :    0.9
  avg tokens per gene name             :   3.33



In [8]:
# ── B3: Cell sentence inspection ──────────────────────────────────────────────
print('=' * 100)
print('  B3: CELL SENTENCE INSPECTION  (true=neuron, side by side)')
print('=' * 100)

def sample_cell_sentences(name, target_label_substr, n=3, top_k=50):
    """Return up to n cell sentences (top-k genes) for cells whose label contains substr."""
    a = adatas[name]
    label_col = LABEL_COLS[name]
    labels = a.obs[label_col].astype(str).values
    mask = np.array([target_label_substr.lower() in lbl.lower() for lbl in labels])
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        return [], []
    chosen = idxs[np.random.default_rng(42).choice(len(idxs), size=min(n, len(idxs)), replace=False)]
    gene_syms = get_gene_symbols(a)
    out = []
    for i in chosen:
        row = a.X[int(i)]
        if hasattr(row, 'toarray'): row = row.toarray().flatten()
        else: row = np.array(row).flatten()
        txt = cell_to_text_dense(row, gene_syms, top_k=top_k)
        out.append(txt)
    return [labels[i] for i in chosen], out

for name in ['brain_neurons', 'brain_normal']:
    print(f'\n--- {name}  (target: any cell with "neuron" in label) ---')
    lbls, sentences = sample_cell_sentences(name, 'neuron', n=3, top_k=50)
    if not lbls:
        print('  (no matching cells found)')
        continue
    for lbl, sent in zip(lbls, sentences):
        print(f'\n  label: {lbl}')
        print(f'  top-50 genes: {sent}')

  B3: CELL SENTENCE INSPECTION  (true=neuron, side by side)

--- brain_neurons  (target: any cell with "neuron" in label) ---

  label: Eccentric medium spiny neuron
  top-50 genes: MALAT1 KCNIP4 ERBB4 RBFOX1 NRXN3 PCDH11X RIMS2 DLG2 ADGRB3 CADM2 SNHG14 SYT1 GALNTL6 FGF14 SOX2-OT CTNNA2 NLGN1 NEGR1 LSAMP TENM2 MEIS2 PRKG1 PDE4D NRXN1 FOXP2 MDGA2 EPHA6 ANKS1B MEG3 PLCB1 PCDH9 PPP3CA KCNH7 ANK3 MACROD2 CSMD3 CSMD1 CAMK2D NKAIN2 LRRC7 NALF1 OLFM3 NRG1 NBEA DPYD PBX3 PPFIA2 AUTS2 FTX UNC5D

  label: CGE interneuron
  top-50 genes: MALAT1 ERBB4 NRXN1 NRXN3 NRG3 CSMD1 CNTNAP2 ANKS1B SNHG14 FGF14 DLG2 GALNTL6 ADGRB3 KAZN ADARB2 GRID2 LRP1B ZBTB20 NBEA PLCB1 CNTN5 SOX2-OT ROBO2 PCDH9 DSCAM LUZP2 SNTG1 DLGAP1 LSAMP AUTS2 CADM2 GRIK2 GRM7 IL1RAPL1 MEG3 NFIB MTUS2 OPCML CNTN4 CHRM3 MDGA2 TCF4 FTX CCSER1 NAV3 LINC03051 TAFA2 MT-CO3 GRIA1 PLEKHA5

  label: Medium spiny neuron
  top-50 genes: MALAT1 KCNIP4 RBFOX1 SNHG14 ROBO1 NRXN3 CSMD3 MACROD2 NLGN1 CNTNAP2 PCDH9 LRRC7 LSAMP ANKS1B RYR2 GRID2 MT-C

In [9]:
# Same comparison for astrocyte / oligodendrocyte
for target in ['astrocyte', 'oligodendrocyte']:
    print('\n' + '=' * 100)
    print(f'  B3 (cont.): cells matching "{target}" in label')
    print('=' * 100)
    for name in ['brain_glia', 'brain_normal']:
        print(f'\n--- {name} ---')
        lbls, sentences = sample_cell_sentences(name, target, n=2, top_k=50)
        if not lbls:
            print('  (no matching cells found)')
            continue
        for lbl, sent in zip(lbls, sentences):
            print(f'\n  label: {lbl}')
            print(f'  top-50 genes: {sent}')


  B3 (cont.): cells matching "astrocyte" in label

--- brain_glia ---

  label: astrocyte
  top-50 genes: MALAT1 TRPM3 PCDH9 PITPNC1 GPM6A LRP1B LSAMP DTNA NRG3 NAV3 OBI1-AS1 GPC5 CDH20 NTM PPP2R2B NEAT1 CADM2 MACF1 NRXN1 LINC03051 NPAS3 CTNND2 PDE4B NTRK2 HPSE2 NAV2 ADGRB3 RYR3 AUTS2 SLC1A3 ERBB4 LINC01088 RORA ADGRL3 MAPK10 SNHG14 MAGI2 ACSBG1 ASTN2 ARHGEF4 NFIB GRID2 PRKCA ANK2 SHROOM3 C2CD5-AS1 GFAP CNTN1 PARD3 NALF1

  label: astrocyte
  top-50 genes: MALAT1 PCDH9 GPC5 NRG3 GPM6A NPAS3 LSAMP ADGRB3 NRXN1 ADGRV1 LRP1B SLC1A2 OBI1-AS1 CTNND2 CADM2 SLC1A3 SOX5 MIR99AHG ZBTB20 TNIK NTM OPHN1 LINC00499 ERBB4 CACNB2 GABRB1 NKAIN3 CTNNA2 QKI ENSG00000271860 PREX2 SLC4A4 LRRC3B TRPM3 DNAH7 ABLIM1 RBMS3 LINC03051 LRRC4C GNAQ RORA CADM1 DTNA MAGI2 ADCY2 NTRK2 ADGRL3 NAV3 ITPR2 CDH20

--- brain_normal ---

  label: astrocyte_protoplasmic
  top-50 genes: CLU AQP4 PTGDS GJA1 CPE CST3 APOE SPARCL1 ATP1B2 GLUL HSPB1 WIF1 ATP1A2 SLC1A3 ADGRV1 LMO3 GPM6A AGT ALDOC ITM2C NTRK2 ETNPPL VIM CKB PLPP3

In [10]:
# ── B4: Training class counts for brain cell types ───────────────────────────
print('=' * 100)
print('  B4: TRAINING-SET CLASS COUNTS  (brain_glia + brain_neurons)')
print('=' * 100)

for name in ['brain_glia', 'brain_neurons']:
    a = adatas[name]
    label_col = LABEL_COLS[name]
    all_counts, kept_counts = label_counts(a, label_col)
    print(f'\n[{name}]  {a.n_obs:,} total cells, label_col={label_col!r}')
    print(f'  raw types: {len(all_counts)},  surviving filter (>={MIN_CELLS_PER_TYPE}): {len(kept_counts)}')
    print(f'  top-20 classes by count:')
    print(kept_counts.head(20).to_string())

    # After applying the MAX_CELLS_PER_TYPE cap (matches main notebook's load_organ)
    capped = kept_counts.clip(upper=MAX_CELLS_PER_TYPE)
    print(f'\n  After MAX_CELLS_PER_TYPE={MAX_CELLS_PER_TYPE} cap: total training cells = {capped.sum():,}')

# Compare to brain_normal labels (the zero-shot target)
print('\n' + '-' * 100)
print('  Brain_normal labels (zero-shot target — what we are trying to predict on):')
print('-' * 100)
a = adatas['brain_normal']
bn_counts = a.obs[LABEL_COLS['brain_normal']].astype(str).value_counts()
print(bn_counts.to_string())

  B4: TRAINING-SET CLASS COUNTS  (brain_glia + brain_neurons)

[brain_glia]  888,263 total cells, label_col='cell_type'
  raw types: 13,  surviving filter (>=100): 13
  top-20 classes by count:
cell_type
oligodendrocyte                           490246
astrocyte                                 155025
oligodendrocyte precursor cell            105734
microglial cell                            88494
fibroblast                                  9156
Bergmann glial cell                         8041
choroid plexus epithelial cell              7689
ependymal cell                              5882
endothelial cell                            5165
committed oligodendrocyte precursor         4720
pericyte                                    3693
central nervous system macrophage           3344
vascular associated smooth muscle cell      1074

  After MAX_CELLS_PER_TYPE=300 cap: total training cells = 3,900

[brain_neurons]  100,000 total cells, label_col='label'
  raw types: 20,  surviving filter (

In [11]:
# =============================================================================
# B5: Validate the proposed gene filter
#   Re-run B3 with EXCLUDE rules applied; compare snRNA-seq vs scRNA-seq cells
# =============================================================================
EXCLUDE_PREFIXES = ('MT-', 'MTRNR', 'RPS', 'RPL', 'LINC', 'MIR',
                    'SNORD', 'SNORA', 'ENSG', 'AC0', 'AC1', 'AL0',
                    'AL1', 'AP0', 'LOC')
EXCLUDE_EXACT    = {'MALAT1', 'NEAT1', 'XIST', 'TSIX', 'KCNQ1OT1',
                    'FTX', 'JPX', 'NORAD', 'HOTAIR'}
EXCLUDE_SUFFIXES = ('-AS1', '-AS2', '-AS3', '-IT1', '-OT1')

def is_keep_gene(g):
    g = str(g)
    if g in EXCLUDE_EXACT: return False
    if g.startswith(EXCLUDE_PREFIXES): return False
    if g.endswith(EXCLUDE_SUFFIXES): return False
    return True

def cell_to_text_dense_filtered(X_row, gene_symbols, top_k=50, apply_filter=True):
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz  = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    if apply_filter:
        keep = np.array([is_keep_gene(gene_symbols[i]) for i in nz])
        nz = nz[keep]
    if len(nz) == 0: return ''
    vals = row[nz]
    if len(nz) > top_k:
        idx = np.argpartition(vals, -top_k)[-top_k:]
        nz  = nz[idx[np.argsort(vals[idx])[::-1]]]
    else:
        nz = nz[np.argsort(vals)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)

def sample_filtered_sentences(name, target_substr, n=2, top_k=50):
    a = adatas[name]
    label_col = LABEL_COLS[name]
    labels = a.obs[label_col].astype(str).values
    mask = np.array([target_substr.lower() in lbl.lower() for lbl in labels])
    idxs = np.where(mask)[0]
    if len(idxs) == 0: return []
    chosen = idxs[np.random.default_rng(42).choice(len(idxs), size=min(n, len(idxs)), replace=False)]
    gene_syms = get_gene_symbols(a)
    out = []
    for i in chosen:
        row = a.X[int(i)]
        if hasattr(row, 'toarray'): row = row.toarray().flatten()
        else: row = np.array(row).flatten()
        raw = cell_to_text_dense_filtered(row, gene_syms, top_k=top_k, apply_filter=False)
        flt = cell_to_text_dense_filtered(row, gene_syms, top_k=top_k, apply_filter=True)
        out.append((labels[i], raw, flt))
    return out

for target in ['astrocyte', 'oligodendrocyte', 'neuron']:
    print('=' * 100)
    print(f'  Filter effect on cells matching "{target}"')
    print('=' * 100)
    for name in ['brain_glia', 'brain_neurons', 'brain_normal']:
        results = sample_filtered_sentences(name, target, n=1, top_k=30)
        if not results: continue
        for lbl, raw, flt in results:
            print(f'\n[{name}]  label: {lbl}')
            print(f'  BEFORE filter (top-30): {raw}')
            print(f'  AFTER  filter (top-30): {flt}')

# Quantitative: how many top-200 genes per dataset pass the filter?
print('\n' + '=' * 100)
print('  Filter-survival rate (% of top-200 expressed genes that pass)')
print('=' * 100)
for name in ['brain_glia', 'brain_neurons', 'brain_normal']:
    a = adatas[name]
    gene_syms = get_gene_symbols(a)
    rng = np.random.default_rng(42)
    idxs = rng.choice(a.n_obs, size=min(30, a.n_obs), replace=False)
    pcts = []
    for i in idxs:
        row = a.X[int(i)]
        if hasattr(row, 'toarray'): row = row.toarray().flatten()
        else: row = np.array(row).flatten()
        nz = np.where(row > 0)[0]
        if len(nz) == 0: continue
        vals = row[nz]
        top = nz[np.argpartition(vals, -min(200,len(vals)))[-min(200,len(vals)):]]
        kept = sum(1 for j in top if is_keep_gene(gene_syms[j]))
        pcts.append(kept / len(top) * 100)
    print(f'  {name:<16} : {np.mean(pcts):>5.1f}% of top-200 genes pass the filter')

  Filter effect on cells matching "astrocyte"

[brain_glia]  label: astrocyte
  BEFORE filter (top-30): MALAT1 TRPM3 PCDH9 PITPNC1 GPM6A LRP1B LSAMP DTNA NRG3 NAV3 OBI1-AS1 GPC5 CDH20 NTM PPP2R2B NEAT1 CADM2 MACF1 NRXN1 LINC03051 NPAS3 CTNND2 PDE4B NTRK2 HPSE2 NAV2 ADGRB3 RYR3 AUTS2 SLC1A3
  AFTER  filter (top-30): TRPM3 PCDH9 GPM6A PITPNC1 LRP1B LSAMP DTNA NAV3 NRG3 GPC5 CDH20 NTM PPP2R2B MACF1 NRXN1 CADM2 CTNND2 NPAS3 ADGRB3 NTRK2 HPSE2 NAV2 PDE4B SLC1A3 RYR3 AUTS2 ERBB4 RORA MAPK10 ADGRL3

[brain_normal]  label: astrocyte_protoplasmic
  BEFORE filter (top-30): CLU AQP4 PTGDS GJA1 CPE CST3 APOE SPARCL1 ATP1B2 GLUL HSPB1 WIF1 ATP1A2 SLC1A3 ADGRV1 LMO3 GPM6A AGT ALDOC ITM2C NTRK2 ETNPPL VIM CKB PLPP3 MT3 DTNA SLC3A2 IFI6 CRYAB
  AFTER  filter (top-30): CLU AQP4 PTGDS GJA1 CPE CST3 APOE SPARCL1 GLUL ATP1B2 HSPB1 ATP1A2 WIF1 SLC1A3 ADGRV1 AGT GPM6A LMO3 NTRK2 ALDOC ITM2C PLPP3 CKB SLC3A2 DTNA VIM MT3 ETNPPL CRYAB SLC1A2
  Filter effect on cells matching "oligodendrocyte"

[brain_glia]  l

In [12]:
# =============================================================================
# B6: Length-normalize snRNA-seq cell sentences; check if markers surface
# =============================================================================
def get_gene_lengths_local(adata, gene_symbols, default=2000.0):
    n = len(gene_symbols)
    if 'feature_length' in adata.var.columns:
        lengths = pd.to_numeric(adata.var['feature_length'], errors='coerce').values.astype(float)
        return np.where((lengths > 0) & np.isfinite(lengths), lengths, default)
    return np.full(n, default, dtype=float)

# Updated filter (more lncRNAs)
EXCLUDE_PREFIXES = ('MT-','MTRNR','RPS','RPL','LINC','MIR','SNORD','SNORA',
                    'SNHG','ENSG','AC0','AC1','AL0','AL1','AP0','LOC')
EXCLUDE_EXACT    = {'MALAT1','NEAT1','XIST','TSIX','KCNQ1OT1','FTX','JPX',
                    'NORAD','HOTAIR','SOX2-OT','TUG1','GAS5','HOTAIRM1','PVT1'}
EXCLUDE_SUFFIXES = ('-AS1','-AS2','-AS3','-AS4','-AS5','-IT1','-IT2',
                    '-OT','-OT1','-OT2','-OT3')

def is_keep_gene2(g):
    g = str(g)
    if g in EXCLUDE_EXACT: return False
    if g.startswith(EXCLUDE_PREFIXES): return False
    if g.endswith(EXCLUDE_SUFFIXES): return False
    return True

def sentence_lnorm(X_row, gene_symbols, gene_lengths, top_k=30, length_normalize=True):
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz  = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    nz = nz[np.array([is_keep_gene2(gene_symbols[i]) for i in nz])]
    if len(nz) == 0: return ''
    vals  = row[nz]
    score = vals / gene_lengths[nz] if length_normalize else vals
    if len(nz) > top_k:
        idx = np.argpartition(score, -top_k)[-top_k:]
        nz  = nz[idx[np.argsort(score[idx])[::-1]]]
    else:
        nz = nz[np.argsort(score)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)


def show_lnorm(name, target_substr, n=1, top_k=30, snrna_seq=False):
    a = adatas[name]
    label_col = LABEL_COLS[name]
    labels = a.obs[label_col].astype(str).values
    mask = np.array([target_substr.lower() in lbl.lower() for lbl in labels])
    idxs = np.where(mask)[0]
    if len(idxs) == 0: return
    chosen = idxs[np.random.default_rng(42).choice(len(idxs), size=min(n, len(idxs)), replace=False)]
    gene_syms = get_gene_symbols(a)
    gene_lens = get_gene_lengths_local(a, gene_syms)
    print(f'\n[{name}]  (snrna_seq={snrna_seq})')
    print(f'  feature_length in var: {"feature_length" in a.var.columns}')
    for i in chosen:
        row = a.X[int(i)]
        if hasattr(row, 'toarray'): row = row.toarray().flatten()
        else: row = np.array(row).flatten()
        lbl = labels[i]
        raw  = sentence_lnorm(row, gene_syms, gene_lens, top_k=top_k, length_normalize=False)
        lnrm = sentence_lnorm(row, gene_syms, gene_lens, top_k=top_k, length_normalize=snrna_seq)
        print(f'  label: {lbl}')
        print(f'  filter only      : {raw}')
        if snrna_seq:
            print(f'  filter + length  : {lnrm}')

for target in ['astrocyte', 'oligodendrocyte', 'neuron']:
    print('=' * 100)
    print(f'  B6 — length-normalized comparison: "{target}"')
    print('=' * 100)
    show_lnorm('brain_glia',    target, n=1, top_k=30, snrna_seq=True)
    show_lnorm('brain_neurons', target, n=1, top_k=30, snrna_seq=True)
    show_lnorm('brain_normal',  target, n=1, top_k=30, snrna_seq=False)

  B6 — length-normalized comparison: "astrocyte"

[brain_glia]  (snrna_seq=True)
  feature_length in var: True
  label: astrocyte
  filter only      : TRPM3 PCDH9 PITPNC1 GPM6A LRP1B LSAMP DTNA NAV3 NRG3 GPC5 CDH20 NTM PPP2R2B MACF1 NRXN1 CADM2 CTNND2 NPAS3 HPSE2 NAV2 PDE4B ADGRB3 NTRK2 SLC1A3 AUTS2 RYR3 ERBB4 MAGI2 MAPK10 ADGRL3
  filter + length  : RYR3 GPM6A GPC5 LSAMP NTM PPP2R2B NAV3 PITPNC1 PDE4B CTNND2 NAV2 RORA NRXN1 SLC1A3 SHROOM3 ACSBG1 CTNNA2 CNTN1 AUTS2 TNRC6A MAGI2 TRPM3 DTNA CADM2 SIL1 RUFY3 SLC39A11 ARHGAP26 AKAP6 LRMDA

[brain_normal]  (snrna_seq=False)
  feature_length in var: False
  label: astrocyte_protoplasmic
  filter only      : CLU AQP4 PTGDS GJA1 CPE CST3 APOE SPARCL1 GLUL ATP1B2 HSPB1 ATP1A2 WIF1 SLC1A3 ADGRV1 AGT GPM6A LMO3 NTRK2 ALDOC ITM2C PLPP3 CKB SLC3A2 DTNA VIM MT3 ETNPPL CRYAB SLC1A2
  B6 — length-normalized comparison: "oligodendrocyte"

[brain_glia]  (snrna_seq=True)
  feature_length in var: True
  label: oligodendrocyte
  filter only      : PCDH9 IL

## Part B Summary

Look at the four diagnostics together to decide where the failure is:

| Diagnostic | If broken, you'd see... | Fix |
|---|---|---|
| **B1 format** | brain_normal uses Ensembl IDs while training uses HGNC symbols (or vice versa) | Convert `var_names` or `feature_name` to a common convention before cell-sentence generation |
| **B2 tokenizer coverage** | brain_normal has noticeably lower `%% genes in vocab` than brain_glia / brain_neurons | Either remap gene names or accept that brain_normal has a different gene panel and pre-process accordingly |
| **B3 cell sentences** | brain_normal neuron sentences are dominated by housekeeping (RPS/RPL/MT-*) with no neuron markers (SNAP25, RBFOX3, GAD1, SLC17A7, etc.) | Increase TOP_K_GENES, or filter out ribosomal/mitochondrial genes before ranking |
| **B4 class counts** | brain_neurons has < 10 surviving classes after the MIN_CELLS filter, or brain_normal labels don't match any training label | Re-balance training data, or add hierarchy mappings so brain_normal labels (e.g. `GABAergic_interneuron_SST`) resolve to a trained ancestor |

The most likely culprit, based on the symptom (model predicts "endothelial cell" / "epithelial cell" / NK on brain cells), is **B1 or B2**: the model can't read the input at all, so it falls back on whatever class has the most generic-token bias. Confirm by comparing the `%% genes in vocab` row across the three brain datasets — if Brain_normal is < 70% while brain_glia / brain_neurons are > 90%, you've found the bug.

## Final recommendations

After running this notebook, you should be able to answer:

1. **Can we safely move lymphoid + myeloid to LAB_CONFIGS?** → Read the Part A coverage table. Acceptable if `lost_if_moved` ≤ 5 leaves of similar functional state. Concerning if it includes whole canonical cell types (e.g. losing all DC subsets).

2. **What's breaking brain_normal?** → Compare B2 numbers across the three datasets. Pick the fix from the table above and implement before re-training.

3. **What changes for next training run?**
   - Move lymphoid + myeloid out of `ORGAN_CONFIGS`, into `LAB_CONFIGS`
   - Apply the brain fix (gene-name remapping or housekeeping filter or both)
   - Retrain CE + HCE end-to-end, re-run the comparison notebook
